<a href="https://colab.research.google.com/github/dhanushmajji1428/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhanushmajji1428/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [21]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

print("\nTrend Direction Distribution")
print(df["trend_direction"].value_counts())

print("\nAverage Position Summary")
print(df["avg_position"].describe())

print("\nImpression Tier Distribution")
print(df["impression_tier"].value_counts())

Dataset Shape: (30000, 44)

Trend Direction Distribution
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Average Position Summary
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64

Impression Tier Distribution
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64


## My Rule

My baseline rule recommends a page for content refresh when it shows a downward traffic trend, poor search visibility, and meaningful search impressions. These signals indicate that the page has visibility but may benefit from updated content.

### Reason Codes

- REFRESH_DECLINE – Downward traffic trend.
- LOW_VISIBILITY – Average search position is poor.
- REFRESH_PRIORITY – Multiple refresh signals are present.

### Action Label

Refresh Content

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
from pathlib import Path

df["baseline_score"] = 0

# Rule 1
df.loc[df["trend_direction"] == "down", "baseline_score"] += 50

# Rule 2
df.loc[df["avg_position"] > 20, "baseline_score"] += 30

# Rule 3
df.loc[df["impressions_90d"] > 1000, "baseline_score"] += 20

df["reason_code"] = "REFRESH_PRIORITY"
df.loc[df["trend_direction"] == "down", "reason_code"] = "REFRESH_DECLINE"

df["action"] = "Refresh Content"

df = df.sort_values("baseline_score", ascending=False)

Path("work/outputs").mkdir(parents=True, exist_ok=True)

df.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("CSV created successfully")

df.head(10)

CSV created successfully


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
29972,content_1db0d204d42f,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6862.0,43541.0,...,1.62,1.86,0.65,good,page_3_5,down,-37.1,100,REFRESH_DECLINE,Refresh Content
35,content_1a28b25c7128,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6877.0,44021.0,...,0.00,12.85,0.00,good,page_3_5,down,-44.6,100,REFRESH_DECLINE,Refresh Content
38,content_dbe82879a406,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,8243.0,55119.0,...,2.02,3.28,3.03,good,page_3_5,down,-51.8,100,REFRESH_DECLINE,Refresh Content
24429,content_c23e6a8b8692,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4950.0,33571.0,...,1.34,29.09,0.67,good,page_3_5,down,-75.1,100,REFRESH_DECLINE,Refresh Content
24404,content_7956cba29f51,client_7f2253d7e2,110.0,1.00,HIGH,1.33,keyword article,transactional,3014.0,19529.0,...,0.00,50.00,0.00,moderate,page_3_5,down,-84.0,100,REFRESH_DECLINE,Refresh Content
24405,content_b9b230d52cfa,client_4e07408562,10.0,0.07,LOW,0.12,keyword article,informational,1479.0,9665.0,...,0.00,0.00,0.00,moderate,page_3_5,down,-48.9,100,REFRESH_DECLINE,Refresh Content
10339,content_fa654994bcce,client_349c41201b,50.0,0.27,LOW,0.10,keyword article,informational,3482.0,21988.0,...,0.00,5.00,0.00,good,page_3_5,down,-59.7,100,REFRESH_DECLINE,Refresh Content
10328,content_d70f12056122,client_4e07408562,20.0,0.00,LOW,0.00,keyword article,transactional,2354.0,14201.0,...,0.00,0.00,0.00,moderate,page_3_5,down,-49.8,100,REFRESH_DECLINE,Refresh Content
10376,content_0eebf153c456,client_3fdba35f04,110.0,0.47,MEDIUM,1.63,keyword article,informational,1312.0,8362.0,...,10.53,19.23,0.00,good,page_3_5,down,-44.0,100,REFRESH_DECLINE,Refresh Content
10399,content_843f4dd5213f,client_4e07408562,10.0,0.13,LOW,0.00,keyword article,transactional,2316.0,14740.0,...,20.00,18.18,0.00,moderate,page_3_5,down,-25.8,100,REFRESH_DECLINE,Refresh Content


The baseline score combines three historical signals:

1. Downward traffic trend
2. Poor average search position
3. High search impressions

Pages with higher scores are ranked higher in the refresh queue. The notebook writes the ranked results to `work/outputs/baseline_action_score.csv`.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [24]:
top20 = df.head(20).copy()

top20["confidence_note"] = "Moderate confidence based on historical search signals."

top20["what_would_make_it_wrong"] = (
    "Seasonality, recent updates, temporary ranking changes, or search intent shifts."
)

top20[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]


,content_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
29972,content_1db0d204d42f,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
35,content_1a28b25c7128,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
38,content_dbe82879a406,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
24429,content_c23e6a8b8692,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
24404,content_7956cba29f51,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
24405,content_b9b230d52cfa,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
10339,content_fa654994bcce,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
10328,content_d70f12056122,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
10376,content_0eebf153c456,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."
10399,content_843f4dd5213f,100,REFRESH_DECLINE,Refresh Content,Moderate confidence based on historical search...,"Seasonality, recent updates, temporary ranking..."


In [26]:
import os

print("Current directory:", os.getcwd())
print("File exists:", os.path.exists("work/outputs/baseline_action_score.csv"))
print("Absolute path:", os.path.abspath("work/outputs/baseline_action_score.csv"))

Current directory: /content/FlyRank
File exists: True
Absolute path: /content/FlyRank/work/outputs/baseline_action_score.csv


In [27]:
import pandas as pd

df_out = pd.read_csv("work/outputs/baseline_action_score.csv")
df_out.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
0,content_1db0d204d42f,client_6208ef0f77,0.0,0.0,LOW,0.00,keyword article,informational,6862.0,43541.0,...,1.62,1.86,0.65,good,page_3_5,down,-37.1,100,REFRESH_DECLINE,Refresh Content
1,content_1a28b25c7128,client_6208ef0f77,0.0,0.0,LOW,0.00,keyword article,informational,6877.0,44021.0,...,0.00,12.85,0.00,good,page_3_5,down,-44.6,100,REFRESH_DECLINE,Refresh Content
2,content_dbe82879a406,client_6208ef0f77,0.0,0.0,LOW,0.00,keyword article,informational,8243.0,55119.0,...,2.02,3.28,3.03,good,page_3_5,down,-51.8,100,REFRESH_DECLINE,Refresh Content
3,content_c23e6a8b8692,client_7f2253d7e2,0.0,0.0,LOW,0.00,keyword article,informational,4950.0,33571.0,...,1.34,29.09,0.67,good,page_3_5,down,-75.1,100,REFRESH_DECLINE,Refresh Content
4,content_7956cba29f51,client_7f2253d7e2,110.0,1.0,HIGH,1.33,keyword article,transactional,3014.0,19529.0,...,0.00,50.00,0.00,moderate,page_3_5,down,-84.0,100,REFRESH_DECLINE,Refresh Content


In [28]:
df_out.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct', 'baseline_score',
       'reason_code', 'action'],
      dtype='object')

## Top-20 Review

The highest-ranked pages were selected because they have the strongest combination of declining performance, lower search visibility, and sufficient impressions.

Confidence is moderate because the baseline rule uses only historical search signals.

Possible false positives include seasonal content, recently updated pages, temporary ranking fluctuations, and changes in search intent.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [25]:
print("Weak Picks")
print("- Seasonal content may be incorrectly flagged.")
print("- Recently updated pages may not yet show improvements.")
print("- Temporary ranking changes may produce false positives.")

print("\nLeakage Check")
print("Historical search signals only.")
print("No future-window information used.")
print("No label-derived features used.")


Weak Picks
- Seasonal content may be incorrectly flagged.
- Recently updated pages may not yet show improvements.
- Temporary ranking changes may produce false positives.

Leakage Check
Historical search signals only.
No future-window information used.
No label-derived features used.


## Weak Picks + Leakage Check

Some pages may be selected even though they do not require a content refresh. Seasonal topics, temporary ranking changes, or recently updated pages may be false positives.

The baseline rule uses only historical search metrics such as trend direction, average position, and impressions. No future-window information or label-derived features were used.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.